# pKaMaP: pKa Measurement by DMS-MaPseq Profiling

This notebook demonstrates a full pKaMaP workflow:

In [1]:
import pkamap
from pkamap.processing import process_json_files, generate_residue_dataframe
from pkamap.fitting import calculate_pka_values, add_structure_annotations
from pkamap.filtering import apply_quality_filters
from pkamap.comparison import summarize_pka, compare_to_nmr
from pkamap.plotting import plot_titration_curves
from pkamap.nmr_reference_C01HK import (
    NMR_REFERENCE,
    MOTIFS_WITH_NMR,
    ALL_LIBRARY_MOTIFS,
    PROTONATION_SITES,
    PROTONATION_SITES_ALL,
    filter_to_protonation_sites,
)

## 1. Configuration

Specify the desired sequences and positions... include path(s) to JSON file(s)

In [1]:
CODES = ["C01HK"]

JSON_FILES = [
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run53.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run54.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run55.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run56.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run44.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run46.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run48.json",
]

# Experimental conditions: (buffer_mM, temp, Mg_mM, color, label)
CONDITIONS = [
    (300, "room", 10, "blue",   "300mM room temp"),
    (300, 25,     10, "red",    "300mM 25C"),
    (50,  "room", 10, "orange", "50mM room temp"),
    (50,  25,     10, "green",  "50mM 25C"),
    (50,  25,     5,  "brown",  "50mM 25C 5mM Mg"),
    (50,  25,     0,  "purple", "50mM 25C 0mM Mg"),
]

## 2. Load & Process Sequencing Data

In [ ]:
df = process_json_files(JSON_FILES, CODES)
df_res = generate_residue_dataframe(df, min_aligned=2000)
print(f"Constructs: {df['name'].nunique()}")
print(f"Total Residues Evaluated: {len(df_res):,}")

## 3. Fit pKa Values & Annotate with Structure

In [ ]:
pka_fits = calculate_pka_values(df_res)
pka_all = add_structure_annotations(pka_fits, df)
print(f"Total fits: {len(pka_all):,}")

## 4. Quality Filtering

Remove pKa fits from residues that have an absent, nonlinear, or noisy response to pH (filtering parameters included in 'filtering.py' file)

In [ ]:
pka_all_unfiltered = pka_all.copy()
pka_all = apply_quality_filters(pka_all)

## 5. Plotting

Inspect individual Henderson-Hasselbalch fits and visually compare conditions
note: takes a lot time to run if you don't filter dataframe for desired motifs!!!

In [ ]:
plot_titration_curves(
    df_res, pka_library, CONDITIONS,
    motif_filter=None,  # or e.g. "GAC&GCC" or ["GAC&GCC", "CCC&GAUG"]
    pka_unfiltered=pka_all_unfiltered,
)

## 6. Select Motifs of Interest (C01HK)

Filter to library to include only motifs inserted into the C01HK library that have residues with experimentally-determined pKa values that have been published

In [ ]:
# All library motifs (including those without published NMR pKa)
pka_library = pka_all[pka_all["motif_sequence"].isin(ALL_LIBRARY_MOTIFS)]

# Only motifs with NMR pKa values
pka_nmr_motifs = pka_all[pka_all["motif_sequence"].isin(MOTIFS_WITH_NMR)]

# Specific residues with NMR-confirmed protonation events
pkamap = filter_to_protonation_sites(pka_nmr_motifs, PROTONATION_SITES)

# All protonation sites (including unpublished)
pkamap_all = filter_to_protonation_sites(pka_library, PROTONATION_SITES_ALL)

print(f"residues with published pKa values: {len(pkamap)}")
print(f"All residues with confirmed protonation events: {len(pkamap_all)}")

## 7. pKa Summary
summarize pKa results

In [ ]:
#summary = summaryize_pka(dataframeofinterest, pka_all_unfiltered, CONDITIONS(from the top of program,)... 'pka_all' is the one that filters for c01hk motifs... pka_all shows all
summary = summarize_pka(pkamap_all, pka_all_unfiltered, CONDITIONS, PROTONATION_SITES_ALL)
summary

## 8. Comparison to Published Values (C01HK)

Compare weighted pKaMaP values to published NMR pKa measurements.

In [ ]:
comparison_stats = compare_to_nmr(
    summary_df=summary,
    nmr_ref_df=NMR_REFERENCE,
    conditions=CONDITIONS,
    plot_combined=True,
)
print(comparison_stats.to_string(index=False))

## 9. Export

In [ ]:
summary.to_csv("summary_pka.csv", index=False)
#comparison_stats.to_csv("comparison_stats.csv", index=False)
#^second line is only for c01hk data!